# 12 · Statistics and maps for the repo

Counts, places and aggregates of the database for the repo's `results/` folder, so anyone opening the repo sees what is in it. No pilot names and no tracks: those stay in Drive.

| File | What |
|---|---|
| `STATS.md` | totals, by year, by source, by country, the sites with most flights, glider classes, pilot ratings and the models' latest scores |
| `coverage_world.png`, `coverage_europe.png`, `coverage_americas.png` | the sites and their 0.25° weather chips, shaded by flights |
| `sites.geojson`, `chips.geojson` | the same as interactive maps (GitHub draws `.geojson` files on a map) |
| `thermal_cells.geojson` | the 5,000 cells of 1 km crossed most often, shaded by how often pilots found a climb (after notebook 09) |
| `stats.json` | the numbers, for scripts |

Run it every 10 to 15 commits, or after a notebook adds data: it writes `database/repo_results/` and downloads a zip to put in the repo. A few minutes.

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive (the id at the end of its link)
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
REPO = "/content/Pre-task-analyzer"

## 2. Mount Drive, get the code

In [ ]:
import pandas as pd
import os, shutil, subprocess
from google.colab import auth, drive
from googleapiclient.discovery import build

auth.authenticate_user()
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    """Folder names from a file up to the top of its drive, joined under root."""
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    """A writable local path for a Drive folder, from its id."""
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):  # in a shared drive
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):  # in my own Drive
        return path_up(folder_id, "/content/drive/MyDrive")
    # Shared with me: write through a shortcut to it in my Drive
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut to it in your Drive "
                             "(right click -> Organize -> Add shortcut), then rerun this cell.")


ROOT = drive_path(DRIVE_FOLDER_ID)
DB = os.path.join(ROOT, "database")
assert os.path.isfile(os.path.join(DB, "sites.parquet")), "Run notebook 06 first: database/sites.parquet is missing"
subprocess.run(["pip", "install", "-q", "cartopy"], check=True)
print("Database in", DB)

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)


def run(*cmd):
    """Run a script and show its output; stopping the cell also stops the script, and a script
    that fails stops Run All."""
    proc = subprocess.Popen(["python3", "-u", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="")
        code = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("Stopped.")
        raise
    if code != 0:
        raise RuntimeError(f"{os.path.basename(cmd[0])} stopped with an error (exit code {code}); see the lines above")
    return code

## 3. Build the statistics and maps

In [ ]:
run(f"{REPO}/scripts/repo_stats.py", DB, "--out", os.path.join(DB, "repo_results"))

## 4. Download them for the repo

A zip of `database/repo_results/`: its files go in the repo's `results/` folder.

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive("/content/repo_results", "zip", os.path.join(DB, "repo_results"))
files.download(zip_path)

## 5. A look

In [ ]:
from IPython.display import Image, Markdown, display
display(Image(os.path.join(DB, "repo_results", "coverage_world.png")))
display(Markdown(open(os.path.join(DB, "repo_results", "STATS.md")).read().split("## Unique tracks")[0]))